# BUILD ONCE — freeze the MISPR environment (no Docker)  ·  v2 (fixed)

Builds the exact Python 3.8 environment the workshop needs and freezes it into one file,
`mispr-env.tar.gz`, that students' Colab sessions download in ~2 minutes.

**This version has all the version fixes baked in** (the molmd repos had moved on to
Python-3.10-only code; these steps pin the last 3.8-compatible versions).

### How to use
1. **Runtime -> Run all.**
2. Stay on the tab and active ~25 minutes.
3. The last cell saves `mispr-env.tar.gz` to your Google Drive.
4. Download it from Drive, upload it to your GitHub **Release**.

Do not edit anything. Just run it.


In [ ]:
#@title Step 1/6 - system build tools (~1 min)
import subprocess
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
sh("apt-get -qq update")
sh("apt-get -qq install -y build-essential gfortran git wget bzip2 > /dev/null")
print("OK")


In [ ]:
#@title Step 2/6 - Miniconda (Python 3.8) at /opt/conda (~1 min)
import os, subprocess
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
if not os.path.exists("/opt/conda/bin/python"):
    sh("wget -q https://repo.anaconda.com/miniconda/Miniconda3-py38_23.11.0-2-Linux-x86_64.sh -O /tmp/mc.sh")
    sh("bash /tmp/mc.sh -b -p /opt/conda")
print(subprocess.run(["/opt/conda/bin/python","--version"],capture_output=True,text=True).stdout.strip())


In [ ]:
#@title Step 3/6 - conda packages incl. MongoDB (~4 min)
import subprocess
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
sh("/opt/conda/bin/conda install -y -c conda-forge "
   "python=3.8 jupyter=1.0.0 jupyterlab=3.4.4 openbabel=3.1.1 conda-pack mongodb=5.0.3")
sh("/opt/conda/bin/conda clean -ya")
print("mongod:", subprocess.run("ls /opt/conda/bin/mongod",shell=True,capture_output=True,text=True).stdout.strip())


In [ ]:
#@title Step 4/6 - pip packages (installed one-by-one to avoid resolver conflict) (~3 min)
import subprocess
PIP = "/opt/conda/bin/pip"
# NOTE: the modern pip in this Miniconda rejects the whole pinned set in one command,
# so we install each package separately (same result, no resolver error).
pkgs = ["doepy==0.0.1","fireworks==1.9.7","llvmlite==0.36.0","numba==0.53.1",
        "numpy==1.23.1","pandas==1.3.4","pymongo==3.12.0","scipy==1.7.1",
        "seaborn==0.11.2","statsmodels==0.13.0"]
for p in pkgs:
    print("installing", p, flush=True)
    subprocess.run(f"{PIP} install -U --no-cache-dir {p}", shell=True, check=True,
                   stdout=subprocess.DEVNULL)
# ruamel MUST be 0.17.21 - newer 0.18+ removed safe_load(), which the pymatgen fork calls.
subprocess.run(f"{PIP} install --no-cache-dir --force-reinstall 'ruamel.yaml==0.17.21'",
               shell=True, check=True, stdout=subprocess.DEVNULL)
print("OK")


In [ ]:
#@title Step 5/6 - MolMD packages, pinned to the last Python-3.8-compatible versions (~8 min)
import os, subprocess
PIP = "/opt/conda/bin/pip"
def run(c, check=False):
    return subprocess.run(c, shell=True, capture_output=True, text=True, check=check)

def clone(url, name):
    d = "/opt/src/" + name
    if not os.path.exists(d):
        run(f"git clone {url} {d}", check=True)   # full clone (need history/tags/branches)
    return d

os.makedirs("/opt/src", exist_ok=True)

# pymatgen: molmd_fix branch (the fork MISPR depends on)
d = clone("https://github.com/molmd/pymatgen.git", "pymatgen")
run(f"cd {d} && git checkout -q molmd_fix", check=True)
run(f"cd {d} && {PIP} install -e . --no-deps", check=True)

# custodian: the gaussian BRANCH has custodian/gaussian/, but its tip uses 3.10 syntax.
# Commit f3bbc83fbc (2024-03-13) is the last one that imports on Python 3.8.
d = clone("https://github.com/molmd/custodian.git", "custodian")
run(f"cd {d} && git fetch -q origin '+refs/heads/*:refs/remotes/origin/*'")
run(f"cd {d} && git checkout -f -q f3bbc83fbc", check=True)
run(f"cd {d} && {PIP} install -e . --no-deps --ignore-requires-python", check=True)

# ParmEd
d = clone("https://github.com/ParmEd/ParmEd.git", "ParmEd")
run(f"cd {d} && {PIP} install -e . --no-deps", check=True)

# mdproptools: v0.0.4 tag (0.0.6 is Python-3.10-only)
d = clone("https://github.com/molmd/mdproptools.git", "mdproptools")
run(f"cd {d} && git fetch -q --tags --force")
if run(f"cd {d} && git checkout -q v0.0.4").returncode != 0:
    sha = run(f"cd {d} && git rev-list -1 --before='2022-06-01' origin/master").stdout.strip()
    run(f"cd {d} && git checkout -q {sha}", check=True)
run(f"cd {d} && {PIP} install -e . --no-deps --ignore-requires-python", check=True)

# mispr: v0.0.4 tag (later releases are Python-3.10-only)
d = clone("https://github.com/molmd/mispr.git", "mispr")
run(f"cd {d} && git fetch -q --tags --force")
if run(f"cd {d} && git checkout -q v0.0.4").returncode != 0:
    sha = run(f"cd {d} && git rev-list -1 --before='2022-09-01' origin/master").stdout.strip()
    run(f"cd {d} && git checkout -q {sha}", check=True)
run(f"cd {d} && {PIP} install -e . --no-deps --ignore-requires-python", check=True)

# dependencies that --no-deps skipped
run(f"{PIP} install --no-cache-dir pubchempy", check=True)
# re-pin scipy + ruamel in case an -e install nudged them
run(f"{PIP} install --no-cache-dir 'scipy==1.7.1'", check=True)
run(f"{PIP} install --no-cache-dir --force-reinstall 'ruamel.yaml==0.17.21'", check=True)
print("OK")


In [ ]:
#@title Step 6/6 - smoke test, then freeze to mispr-env.tar.gz (~3 min)
import subprocess
def sh(c):
    print("$", c, flush=True); subprocess.run(c, shell=True, check=True)

test = ("import pymatgen, custodian, parmed, mdproptools, mispr, fireworks, numba, seaborn; "
        "from mispr.gaussian.workflows.base.esp import get_esp_charges; "
        "from mispr.gaussian.utilities.fw_utilities import run_fake_gaussian; "
        "from mispr.lammps.utilities.utilities import run_fake_lammps; "
        "from mispr.lammps.workflows.base import lammps_workflow; "
        "print('IMPORTS OK')")
sh(f'/opt/conda/bin/python -c "{test}"')

# --ignore-missing-files: the pip --force-reinstall of ruamel left conda's file
# records inconsistent; this flag lets conda-pack proceed (the working files are packed).
sh("/opt/conda/bin/conda-pack -p /opt/conda -o /tmp/env.tar.gz "
   "--ignore-editable-packages --ignore-missing-files --force")
sh("rm -rf /tmp/stage && mkdir -p /tmp/stage/opt/conda /tmp/stage/opt/src")
sh("tar -xzf /tmp/env.tar.gz -C /tmp/stage/opt/conda")
sh("cp -r /opt/src/. /tmp/stage/opt/src/")
sh("cd /tmp/stage && tar -czf /content/mispr-env.tar.gz opt")
import os
print("\nSIZE:", round(os.path.getsize('/content/mispr-env.tar.gz')/1e9, 2), "GB")
print("BUILT: /content/mispr-env.tar.gz")


In [ ]:
#@title Save the file to your Google Drive
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
os.makedirs('/content/drive/MyDrive/mispr_build', exist_ok=True)
shutil.copy('/content/mispr-env.tar.gz', '/content/drive/MyDrive/mispr_build/mispr-env.tar.gz')
print("Saved to Google Drive -> MyDrive/mispr_build/mispr-env.tar.gz")
print("Next: download it from Drive, then upload to your GitHub Release.")
